In [1]:
!pip install pandas scikit-learn joblib -q

import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
import joblib

In [2]:
!wget https://archive.ics.uci.edu/static/public/327/phishing+websites.zip -O phishing.zip
!unzip -o phishing.zip
!ls

--2026-06-23 11:17:57--  https://archive.ics.uci.edu/static/public/327/phishing+websites.zip
Resolving archive.ics.uci.edu (archive.ics.uci.edu)... 128.195.10.252
Connecting to archive.ics.uci.edu (archive.ics.uci.edu)|128.195.10.252|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: unspecified
Saving to: ‘phishing.zip’

phishing.zip            [    <=>             ] 987.48K  1.25MB/s    in 0.8s    

2026-06-23 11:17:59 (1.25 MB/s) - ‘phishing.zip’ saved [1011184]

Archive:  phishing.zip
 extracting: .old.arff               
 extracting: Training Dataset.arff   
 extracting: Phishing Websites Features.docx  
'Phishing Websites Features.docx'   sample_data
 phishing.zip			   'Training Dataset.arff'


In [3]:
from scipy.io import arff
import pandas as pd

data, meta = arff.loadarff('Training Dataset.arff')
df = pd.DataFrame(data)

print(df.shape)
df.head()

(11055, 31)


,having_IP_Address,URL_Length,Shortining_Service,having_At_Symbol,double_slash_redirecting,Prefix_Suffix,having_Sub_Domain,SSLfinal_State,Domain_registeration_length,Favicon,...,popUpWidnow,Iframe,age_of_domain,DNSRecord,web_traffic,Page_Rank,Google_Index,Links_pointing_to_page,Statistical_report,Result
0,b'-1',b'1',b'1',b'1',b'-1',b'-1',b'-1',b'-1',b'-1',b'1',...,b'1',b'1',b'-1',b'-1',b'-1',b'-1',b'1',b'1',b'-1',b'-1'
1,b'1',b'1',b'1',b'1',b'1',b'-1',b'0',b'1',b'-1',b'1',...,b'1',b'1',b'-1',b'-1',b'0',b'-1',b'1',b'1',b'1',b'-1'
2,b'1',b'0',b'1',b'1',b'1',b'-1',b'-1',b'-1',b'-1',b'1',...,b'1',b'1',b'1',b'-1',b'1',b'-1',b'1',b'0',b'-1',b'-1'
3,b'1',b'0',b'1',b'1',b'1',b'-1',b'-1',b'-1',b'1',b'1',...,b'1',b'1',b'-1',b'-1',b'1',b'-1',b'1',b'-1',b'1',b'-1'
4,b'1',b'0',b'-1',b'1',b'1',b'-1',b'1',b'1',b'-1',b'1',...,b'-1',b'1',b'-1',b'-1',b'0',b'-1',b'1',b'1',b'1',b'1'


In [4]:
# Convert all byte-string columns to actual integers
for col in df.columns:
    df[col] = df[col].apply(lambda x: int(x.decode('utf-8')))

print(df.dtypes.head())
df.head()

having_IP_Address           int64
URL_Length                  int64
Shortining_Service          int64
having_At_Symbol            int64
double_slash_redirecting    int64
dtype: object


,having_IP_Address,URL_Length,Shortining_Service,having_At_Symbol,double_slash_redirecting,Prefix_Suffix,having_Sub_Domain,SSLfinal_State,Domain_registeration_length,Favicon,...,popUpWidnow,Iframe,age_of_domain,DNSRecord,web_traffic,Page_Rank,Google_Index,Links_pointing_to_page,Statistical_report,Result
0,-1,1,1,1,-1,-1,-1,-1,-1,1,...,1,1,-1,-1,-1,-1,1,1,-1,-1
1,1,1,1,1,1,-1,0,1,-1,1,...,1,1,-1,-1,0,-1,1,1,1,-1
2,1,0,1,1,1,-1,-1,-1,-1,1,...,1,1,1,-1,1,-1,1,0,-1,-1
3,1,0,1,1,1,-1,-1,-1,1,1,...,1,1,-1,-1,1,-1,1,-1,1,-1
4,1,0,-1,1,1,-1,1,1,-1,1,...,-1,1,-1,-1,0,-1,1,1,1,1


In [5]:
# Separate features (X) from the target label (y)
X = df.drop('Result', axis=1)
y = df['Result']

print("Features shape:", X.shape)
print("Label distribution:")
print(y.value_counts())

Features shape: (11055, 30)
Label distribution:
Result
 1    6157
-1    4898
Name: count, dtype: int64


In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Training set:", X_train.shape)
print("Test set:", X_test.shape)

Training set: (8844, 30)
Test set: (2211, 30)


In [7]:
model = RandomForestClassifier(
    n_estimators=100,
    max_depth=None,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)
print("Training complete")

Training complete


In [8]:
predictions = model.predict(X_test)

accuracy = accuracy_score(y_test, predictions)
print(f"Accuracy: {accuracy:.4f}")
print()
print("Classification Report:")
print(classification_report(y_test, predictions, target_names=['Phishing', 'Legitimate']))

Accuracy: 0.9742

Classification Report:
              precision    recall  f1-score   support

    Phishing       0.98      0.96      0.97       980
  Legitimate       0.97      0.98      0.98      1231

    accuracy                           0.97      2211
   macro avg       0.97      0.97      0.97      2211
weighted avg       0.97      0.97      0.97      2211



In [9]:
import numpy as np

# Find the actual phishing sites that were misclassified as legitimate
misclassified_mask = (y_test.values == -1) & (predictions == 1)
missed_phishing = X_test[misclassified_mask]

print(f"Missed phishing sites: {misclassified_mask.sum()} out of {(y_test == -1).sum()}")
print()
print("Feature averages for MISSED phishing sites vs. all phishing sites:")
print(pd.DataFrame({
    'Missed phishing (avg)': missed_phishing.mean(),
    'All phishing (avg)': X_test[y_test == -1].mean()
}))

Missed phishing sites: 38 out of 980

Feature averages for MISSED phishing sites vs. all phishing sites:
                             Missed phishing (avg)  All phishing (avg)
having_IP_Address                        -0.052632            0.167347
URL_Length                               -0.710526           -0.684694
Shortining_Service                        0.842105            0.785714
having_At_Symbol                          0.736842            0.671429
double_slash_redirecting                  0.947368            0.787755
Prefix_Suffix                            -1.000000           -1.000000
having_Sub_Domain                         0.052632           -0.223469
SSLfinal_State                            0.368421           -0.460204
Domain_registeration_length              -0.210526           -0.077551
Favicon                                   0.894737            0.646939
port                                      0.894737            0.720408
HTTPS_token                               0

In [10]:
joblib.dump(model, 'phishing_model.pkl')

# Also save the feature column order — the dashboard will need this
# to make sure new data is structured exactly the same way
joblib.dump(list(X.columns), 'phishing_feature_columns.pkl')

from google.colab import files
files.download('phishing_model.pkl')
files.download('phishing_feature_columns.pkl')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>